In [ ]:
#GROUP DESCRIPTIVES
"""This script creates descriptive tables for the three cognitive groups and the dementia group"""

import os
import numpy as np
import pandas as pd

DATA = "cv_results_all_tests_merged_with_groups.csv"
OUT  = "descriptives"
os.makedirs(OUT, exist_ok=True)

GROUPS     = ["HC", "Borderline", "IC"]          # mutually exclusive cognitive groups
FEMALE     = 0                                      # UKB field 31: 0 = Female, 1 = Male
AGE_BINS   = [-np.inf, 50, 60, 70, 80, np.inf]      # lowest bin open so participants <40 are kept
AGE_LABELS = ["<50", "50–59", "60–69", "70–79", "80+"]

df = pd.read_csv(DATA)
for c in ["age_at_test", "age_at_dx", "age_at_followup_end", "z_score"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"{len(df):,} test rows | {df['eid'].nunique():,} participants")

In [ ]:
#GROUPS, VISITS, BASELINE

# Visit level: age and number of impaired tests per participant per visit
visit = (df.groupby(["eid", "instance"])
           .agg(age=("age_at_test", "min"), n_impaired=("impaired", "sum"))
           .reset_index())

# Group membership
max_imp = visit.groupby("eid")["n_impaired"].max()
eids = {
    "HC":        set(df.loc[df["hc"] == 1, "eid"]),
    "Borderline": set(max_imp.index[max_imp == 1]),
    "IC":        set(df.loc[df["ic"] == 1, "eid"]),
    "Dementia":   set(df.loc[df["dem_all"] == 1, "eid"]),
}
df["group"] = df["eid"].map({e: g for g in GROUPS for e in eids[g]})

# Check that the three cognitive groups don't overlap
n_grouped = df.loc[df["group"].notna(), "eid"].nunique()
assert sum(len(eids[g]) for g in GROUPS) == n_grouped, "cognitive groups overlap"
print({g: f"{len(e):,}" for g, e in eids.items()},
      f"| ungrouped: {df['eid'].nunique() - n_grouped:,}")

# Baseline: one row per participant at their chronologically earliest visit
base = (df.sort_values(["eid", "age_at_test", "instance"])
          .drop_duplicates("eid").set_index("eid"))
base["age_bin"] = pd.cut(base["age_at_test"], AGE_BINS, labels=AGE_LABELS, right=False)
base["obs_yrs"] = base["age_at_followup_end"] - base["age_at_test"]   # baseline -> end of follow-up

# Age at the first visit with >= 2 impaired tests
first_ic_age = (visit[visit["n_impaired"] >= 2]
                 .sort_values(["age", "instance"])
                 .drop_duplicates("eid").set_index("eid")["age"])

In [ ]:
#TABLE by group
"""One column per group. Test-based ages and spans come from all test rows; 
sex, education and observation time come from the baseline row. Observation 
time runs from baseline to end of follow-up and excludes prevalent dementia. """

def mean_sd(s):
    s = pd.Series(s).dropna()
    return f"{s.mean():.1f} ({s.std():.1f})" if len(s) else ""

def median_iqr(s):
    s = pd.Series(s).dropna()
    if s.empty:
        return ""
    q1, q3 = s.quantile([.25, .75])
    return f"{s.median():.1f} [{q1:.1f}–{q3:.1f}]"

def value_range(s):
    s = pd.Series(s).dropna()
    return f"{s.min():.1f}–{s.max():.1f}" if len(s) else ""

def table1(label):
    ids  = eids[label]
    rows = df[df["eid"].isin(ids)]
    b    = base[base.index.isin(ids)]
    span = rows.groupby("eid").agg(first=("age_at_test", "min"), last=("age_at_test", "max"),
                                    visits=("instance", "nunique"))
    obs  = b.loc[b["prevalent"] != 1, "obs_yrs"]

    row = {
        "n participants":                        len(ids),
        "n tests":                               len(rows),
        "Baseline age, mean (SD)":               mean_sd(span["first"]),
        "Last test age, mean (SD)":              mean_sd(span["last"]),
        "First–last test span (yrs), mean (SD)": mean_sd(span["last"] - span["first"]),
        "Visits, mean (SD)":                     mean_sd(span["visits"]),
        "% Female":                              round(100 * (b["Sex"] == FEMALE).mean(), 1),
        "Education, mode":                       b["Edu"].mode().iloc[0],
        "Observation (yrs), median [IQR]":       median_iqr(obs),
        "Observation (yrs), range":              value_range(obs),
    }
    if label in GROUPS:
        row["% with dementia"] = round(100 * len(ids & eids["Dementia"]) / len(ids), 1)
    if label == "IC":
        row["Age at first IC visit, mean (SD)"] = mean_sd(first_ic_age.reindex(list(ids)))
    if label == "Dementia":
        inc   = b[b["prevalent"] == 0]
        to_dx = (inc["age_at_dx"] - inc["age_at_test"]).loc[lambda s: s >= 0]
        row |= {
            "Age at diagnosis, mean (SD)":                mean_sd(b["age_at_dx"]),
            "n incident":                                 len(to_dx),
            "Baseline test → diagnosis (yrs), mean (SD)": mean_sd(to_dx),
            "Baseline test → diagnosis (yrs), range":     value_range(to_dx),
        }
    return pd.Series(row, name=label)

t1 = pd.concat([table1(g) for g in GROUPS + ["Dementia"]], axis=1)
t1.to_csv(f"{OUT}/table1_by_group.csv")
t1

In [ ]:
#VISIT STRUCTURE

"""How many visits each participant has (whole sample), and what share of each group's test rows comes from each instance."""

n_visits = df.groupby("eid")["instance"].nunique()
visits = n_visits.value_counts().sort_index().rename_axis("n visits").to_frame("n participants")
visits["%"] = (100 * visits["n participants"] / len(n_visits)).round(1)
print(f"Visits per participant: mean {n_visits.mean():.1f} (SD {n_visits.std():.1f}), "
      f"median {n_visits.median():.0f}, range {n_visits.min()}–{n_visits.max()}")

inst_by_group = (pd.crosstab(df["group"], df["instance"], normalize="index") * 100).round(1).reindex(GROUPS)

visits.to_csv(f"{OUT}/instance_distribution.csv")
inst_by_group.to_csv(f"{OUT}/instance_by_group.csv")
display(visits, inst_by_group)

In [ ]:
#PREVALENCE BY BASLINE AGE, EDU, SEX

"""Within each level of a baseline variable, the n and % of participants in each cognitive group."""

def prevalence_by(var):
    n = pd.crosstab(base[var], base["group"]).reindex(columns=GROUPS, fill_value=0)
    out = pd.DataFrame({"n total": n.sum(axis=1)})
    for g in GROUPS:
        out[f"n {g}"] = n[g]
        out[f"% {g}"] = (100 * n[g] / out["n total"]).round(2)
    return out

for var in ["age_bin", "Edu", "Sex"]:
    tab = prevalence_by(var)
    tab.to_csv(f"{OUT}/prevalence_by_{var.lower()}.csv")
    display(tab)

In [ ]:
#GROUPS PREVALENCE BY VISIT

""""Status is assessed at each visit separately: IC means ≥2 impaired tests at that visit, 
and Borderline means exactly 1. This differs from the longitudinal groups above. A participant 
can be Borderline at one visit and IC at a later one."""


v = visit.assign(
    age_bin=pd.cut(visit["age"], AGE_BINS, labels=AGE_LABELS, right=False),
    IC=visit["n_impaired"] >= 2,
    Bord=visit["n_impaired"] == 1,
)
xs = (v.groupby(["instance", "age_bin"], observed=False)
        .agg(n_total=("eid", "size"), n_IC=("IC", "sum"), n_Bord=("Bord", "sum"))
        .reset_index())
xs["% IC"]  = (100 * xs["n_IC"]  / xs["n_total"]).round(2)
xs["% Bord"] = (100 * xs["n_Bord"] / xs["n_total"]).round(2)
xs = xs.merge(v.groupby("instance")["age"].mean().round(1).rename("mean age (instance)"), on="instance")

xs.to_csv(f"{OUT}/prevalence_ic_bord_by_instance_age.csv", index=False)
display(xs.pivot(index="age_bin", columns="instance", values="% IC"),
        xs.pivot(index="age_bin", columns="instance", values="% Bord"))

In [ ]:
#Z-SCORE DISTRIBUTIONS BY TEST AND GROUP

"""Mean, SD, median, quartiles and range of the normative z-scores for each test within each cognitive group. Tests without z-scores, such as PMem, drop out."""

def q(p):
    return lambda s: s.quantile(p)

zt = (df.dropna(subset=["z_score", "group"])
        .groupby(["test_name", "group"])["z_score"]
        .agg(mean="mean", sd="std", median="median", q25=q(.25), q75=q(.75), min="min", max="max")
        .round(2)
        .unstack("group")
        .swaplevel(axis=1)[GROUPS])

zt.to_csv(f"{OUT}/z_table.csv")
zt

In [ ]:
#CHECKS


#Short observation windows check

""" Lists the participants with the shortest baseline → end-of-follow-up windows, to show which baseline 
instances drive the minimum, plus the shortest baseline → diagnosis intervals for incident dementia."""

END_OF_FOLLOWUP = 2025 + 8/12          # same cut-off as notebook 05b
N_SHOW = 20

qc = base.loc[base["prevalent"] != 1,
              ["instance", "age_at_test", "age_at_followup_end", "obs_yrs",
               "dem_all", "age_at_dx", "group"]].copy()
qc["baseline_calendar_yr"] = (END_OF_FOLLOWUP - qc["obs_yrs"]).round(1)   # approx: age_at_test is whole years
qc["test_to_dx"] = qc["age_at_dx"] - qc["age_at_test"]

shortest = qc.nsmallest(N_SHOW, "obs_yrs")
print("Baseline instance of shortest windows:", shortest["instance"].value_counts().sort_index().to_dict())
display(shortest)
display(qc[(qc["dem_all"] == 1) & (qc["test_to_dx"] >= 0)].nsmallest(10, "test_to_dx"))


#Education and dementia relationship check

"""Shows the education distribution by dementia status within baseline age bands, then an age- 
and sex-adjusted logistic model with Edu 6 as the reference. Prevalent dementia is excluded."""

import statsmodels.formula.api as smf

pp = base[base["prevalent"] != 1].reset_index()
band = pd.cut(pp["age_at_test"], [-np.inf, 50, 60, 65, 70, np.inf], right=False)
display((pd.crosstab([band, pp["dem_all"]], pp["Edu"], normalize="index") * 100).round(1))

m = smf.logit("dem_all ~ C(Edu, Treatment(6)) + age_at_test + C(Sex)", data=pp).fit(disp=0)
m.summary().tables[1]